##  Step 1 — Installation

In [ ]:
# ── TIER 0: Zero-install mode — works immediately with no pip installs ────────
# The pipeline runs fully with stdlib only. Run all cells below without pip.

# ── TIER 1: Full ML pipeline (recommended for Google Colab) ──────────────────
# Uncomment and run the block below for transformer-based models:

%pip install transformers torch sentencepiece tokenizers \
           sentence-transformers faiss-cpu rank-bm25 \
             spacy nltk shap pandas numpy scikit-learn \
             matplotlib seaborn plotly requests -q

%python -m spacy download en_core_web_sm -q

import nltk
# nltk.download('stopwords', quiet=True)
# nltk.download('punkt', quiet=True)

print('✓ Ready to run (zero-install / stdlib mode)')
print('  Uncomment the pip block above for full transformer support.')

##  Step 2 — Clone Repository

In [ ]:
import os

# If running from a cloned repo, set the path:
# REPO_PATH = '/content/geosentiafake'   # Colab
# REPO_PATH = '.'                        # Local

# For Colab: upload the geosentiafake/ folder or clone from GitHub:
# !git clone https://github.com/YOUR_USERNAME/geosentiafake.git
# REPO_PATH = '/content/geosentiafake'

REPO_PATH = '/Users/aradhyasharma/Desktop/final yr research ppr/geosentifake/files (1)/geosentiafake/GeoSentiFake_Notebook.ipynb'  # adjust to your path
os.chdir(REPO_PATH)
print(f'Working directory: {os.getcwd()}')
print('Files:', [f for f in os.listdir('.') if not f.startswith('.')])

##  Step 3 — Import Pipeline

In [ ]:
import sys
sys.path.insert(0, REPO_PATH)

from utils.config    import Config
from utils.logger    import setup_logger
from utils.display   import PipelineDisplay
from geosentiafake   import GeoSentiFakePipeline, DEMO_ARTICLES

config  = Config()
logger  = setup_logger(verbose=False)
pipeline = GeoSentiFakePipeline(config, logger)
print('✓ Pipeline initialised')

##  Step 4 — Run Demo Articles

In [ ]:
results = pipeline.run_batch(DEMO_ARTICLES)

##  Step 5 — Visualise Results

In [ ]:
import json

# Save results
with open('results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('✓ Saved results.json')

# Generate charts
from utils.visualiser import generate_all_charts
generate_all_charts('results.json', output_dir='charts')

In [ ]:
# Display charts inline (Jupyter)
from IPython.display import Image, display
import os

for chart in ['ess_comparison.png','emotion_heatmap.png','vad_scatter.png','feature_importance.png']:
    path = os.path.join('charts', chart)
    if os.path.exists(path):
        print(f'\n── {chart} ──')
        display(Image(path))

##  Step 6 — Analyse Single Article (Custom Input)

In [ ]:
# ── Analyse your own article ──────────────────────────────────────────────────
my_article = {
    'id':           'custom_001',
    'title':        'PASTE YOUR HEADLINE HERE',
    'text':         'Paste the full article body here...',
    'source':       'example.com',
    'date':         '2024-11-01',
    'gdelt_region': 'South Asia',   # or: Middle East, Eastern Europe, etc.
}

result = pipeline.run_article(my_article)

# Pretty print key outputs
print(f"\n{'═'*60}")
print(f"  LABEL      : {result['label']}  ({result['confidence']:.1%} confidence)")
print(f"  ESS SCORE  : {result['ess_score']:.1f} / 100  [{result['ess_band']}]")
print(f"  GSM        : {result['gsm']}×  ({result['gsm_level']})")
print(f"  EIS        : {result['eis']:.3f}")
print(f"  DBS        : {result['dbs']:.3f}")
print(f"  CDS        : {result['cds']:.3f}")
print(f"  STANCE     : {result['stance']} ({result['stance_conf']:.1%})")
print(f"  CREDIBILITY: {result['credibility']:.2f}")
print(f"  EMOTION    : {result['dominant_emotion'].upper()}")
print(f"  VAD        : V={result['vad']['valence']:+.3f}  A={result['vad']['arousal']:.3f}  D={result['vad']['dominance']:+.3f}")
print(f"{'─'*60}")
print(f"  VERDICT:\n  {result['verdict_text']}")
print(f"{'═'*60}")

##  Step 7 — Batch Processing from CSV

In [ ]:
# Load from CSV
from geosentiafake import load_articles_from_csv

# CSV format: id, title, text, source, date, gdelt_region
csv_results = pipeline.run_batch(
    load_articles_from_csv('sample_articles.csv')
)

# Export to JSON
with open('csv_results.json', 'w') as f:
    json.dump(csv_results, f, indent=2)
print('\n✓ CSV batch results saved to csv_results.json')

##  Step 8 — ESS Deep Dive

In [ ]:
# ESS sub-score breakdown
print(f"{'Article':<40} {'EIS':>6} {'DBS':>6} {'CDS':>6} {'GSM':>5} {'ESS':>7} {'Band':<10}")
print('─' * 90)
for r in results:
    print(f"{r['title'][:40]:<40} "
          f"{r['eis']:>6.3f} {r['dbs']:>6.3f} {r['cds']:>6.3f} "
          f"{r['gsm']:>5.1f} {r['ess_score']:>7.1f} {r['ess_band']:<10}")

## Step 9 — Emotion Profile Deep Dive

In [ ]:
# Emotion comparison across articles
emotions = ['fear','anger','disgust','sadness','surprise','joy','trust','anticipation']
print(f"{'Article':<35} " + ' '.join(f"{e[:5]:>6}" for e in emotions))
print('─' * 90)
for r in results:
    emos = r['emotions']
    row = f"{r['title'][:35]:<35} "
    row += ' '.join(f"{emos.get(e,0):>6.3f}" for e in emotions)
    print(row)

## Step 10 — Configuration Tuning

All hyperparameters are in `utils/config.py`. Key values to tune:

In [ ]:
# ── Tune ESS coefficients ─────────────────────────────────────────────────────
custom_config = Config()
custom_config.alpha = 0.45   # higher weight on emotional intensity
custom_config.beta  = 0.30   # lower weight on directional bias
custom_config.gamma = 0.25   # unchanged contextual deviation

# Add custom regions as active conflict zones
custom_config.active_conflict_regions.append('Central Asia')

# Rebuild pipeline with new config
custom_pipeline = GeoSentiFakePipeline(custom_config, logger)
custom_result   = custom_pipeline.run_article(DEMO_ARTICLES[0])
print(f"Custom ESS: {custom_result['ess_score']:.1f}  (original: {results[0]['ess_score']:.1f})")

## Step 11 — Enable Live Web Retrieval (Optional)

For production use with real-time evidence retrieval, set API keys:

In [ ]:
import os

# Option A: SerpAPI (https://serpapi.com — free tier available)
# os.environ['SERPAPI_KEY'] = 'your_serpapi_key_here'

# Option B: Google Custom Search API
# os.environ['GOOGLE_CSE_KEY'] = 'your_google_api_key'
# os.environ['GOOGLE_CSE_CX']  = 'your_search_engine_id'

# After setting keys, rebuild the pipeline:
# pipeline = GeoSentiFakePipeline(config, logger)
# Results will now use live web retrieval instead of the offline knowledge base.

print('Set SERPAPI_KEY or GOOGLE_CSE_KEY env var for live retrieval.')
print(f"Current retrieval mode: {results[0].get('retrieval_mode', 'offline_kb')}")

## Step 12 — Enable Full Transformer Models (Colab GPU)

Upgrade from heuristic to real transformer inference:

In [ ]:
# After running: !pip install transformers torch
# Uncomment the model loading lines in each pipeline module:

# pipeline/fake_detector.py  → Uncomment FakeBERT pipeline() call
#   model: 'jy46604790/Fake-News-Bert-Detect' (binary, public)
#   OR fine-tune bert-base-uncased on ISOT+LIAR for 3-class

# pipeline/emotion_engine.py → Uncomment GoEmotions pipeline() call
#   model: 'SamLowe/roberta-base-go_emotions' (public, ~500MB)

# pipeline/stance_detector.py → Uncomment DeBERTa pipeline() call
#   model: fine-tuned deberta-v3-base on FNC-1 + FEVER

# Check GPU availability:
try:
    import torch
    print(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
except ImportError:
    print('torch not installed — run pip install torch')

##  Step 13 — Export Full Results

In [ ]:
import csv

# Export to CSV for analysis
fieldnames = [
    'article_id','title','source','date','label','confidence',
    'ess_score','ess_band','eis','dbs','cds','gsm','gsm_level',
    'dominant_emotion','stance','credibility','latency_ms'
]
with open('results_export.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in results:
        writer.writerow({k: r.get(k,'') for k in fieldnames})

print('✓ Exported to results_export.csv')

# Download in Colab
try:
    from google.colab import files
    files.download('results_export.csv')
    files.download('results.json')
except ImportError:
    print('  (Not in Colab — files saved locally)')